# 衛星データ特徴量の作成

Sentinel-2 および VIIRS の TIF ファイルから特徴量を計算し、`train_tabular.csv` に追加する。

## 処理内容
- Sentinel-2 の 12 バンド（生値）の統計量
- Sentinel-2 から計算した各種指数の統計量
- VIIRS の `avg_rad` バンドの統計量

## 集約方法
パーセンタイル 0,10,20,30,40,50,60,70,80,90,100、平均、標準偏差

## カラム名
`{band_name}_{集約方法}` 例: `B1_pct0`, `NDBI_mean`, `avg_rad_std`

In [1]:
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')

In [2]:
# DATA_DIR = Path('../data/train_dataset_82ddf14911a54c729380209510ae25ac')
# COMPOSITE_DIR = DATA_DIR / 'train_composite'
# CSV_PATH = DATA_DIR / 'train_tabular.csv'
# OUTPUT_CSV_PATH = DATA_DIR / 'train_tabular-add-features.csv'

DATA_DIR = Path('../data/evaluation_dataset_88c8c1d9376e4d9b8034f77ca19939b0')
COMPOSITE_DIR = DATA_DIR / 'evaluation_composite'
CSV_PATH = DATA_DIR / 'evaluation_tabular_no_target.csv'
OUTPUT_CSV_PATH = DATA_DIR / 'evaluation_tabular_no_target-add-features.csv'

PERCENTILES = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

df = pd.read_csv(CSV_PATH)
print(f'CSV shape: {df.shape}')
print(f'Sentinel-2 unique files: {df["sentinel2_tiff_file_name"].nunique()}')
print(f'VIIRS unique files: {df["viirs_tiff_file_name"].nunique()}')

CSV shape: (1024, 22)
Sentinel-2 unique files: 1024
VIIRS unique files: 1024


In [3]:
def safe_divide(a, b):
    """ゼロ除算を NaN で置換する除算"""
    with np.errstate(invalid='ignore', divide='ignore'):
        result = np.where(b != 0, a / b, np.nan)
    return result.astype(np.float64)


def compute_stats(arr, band_name):
    """1次元配列からパーセンタイル・平均・標準偏差を計算し、辞書で返す"""
    valid = arr[np.isfinite(arr)]
    stats = {}
    if len(valid) == 0:
        for p in PERCENTILES:
            stats[f'{band_name}_pct{p}'] = np.nan
        stats[f'{band_name}_mean'] = np.nan
        stats[f'{band_name}_std'] = np.nan
    else:
        for p in PERCENTILES:
            stats[f'{band_name}_pct{p}'] = float(np.percentile(valid, p))
        stats[f'{band_name}_mean'] = float(np.mean(valid))
        stats[f'{band_name}_std'] = float(np.std(valid))
    return stats

In [4]:
def process_sentinel2(tif_path):
    """
    Sentinel-2 TIF を読み込み、生バンドの統計量および各種指数の統計量を返す。
    
    バンド順: B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12
    """
    with rasterio.open(tif_path) as src:
        data = src.read().astype(np.float64)  # shape: (12, H, W)

    band_names = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']
    bands = {name: data[i].ravel() for i, name in enumerate(band_names)}

    result = {}

    # --- 生バンドの統計量 ---
    for name, arr in bands.items():
        result.update(compute_stats(arr, name))

    # --- 指数の計算 ---
    B1  = bands['B1']
    B2  = bands['B2']
    B3  = bands['B3']
    B4  = bands['B4']
    B7  = bands['B7']
    B8  = bands['B8']
    B8A = bands['B8A']
    B11 = bands['B11']
    B12 = bands['B12']

    # 仮説1: 都市化・建設物密度
    NDBI = safe_divide(B11 - B8, B11 + B8)
    result.update(compute_stats(NDBI, 'NDBI'))

    UI = safe_divide(B7 - B4, B7 + B4)
    result.update(compute_stats(UI, 'UI'))

    # 仮説2: 植生
    NDVI = safe_divide(B8 - B4, B8 + B4)
    result.update(compute_stats(NDVI, 'NDVI'))

    EVI_denom = B8 + 6.0 * B4 - 7.5 * B2 + 1.0
    EVI = safe_divide(2.5 * (B8 - B4), EVI_denom)
    result.update(compute_stats(EVI, 'EVI'))

    # 仮説1: IBI（NDVIとMNDWIを利用するため、ここで計算）
    MNDWI = safe_divide(B3 - B11, B3 + B11)  # 先に計算
    IBI_num = NDBI - (NDVI + MNDWI) / 2.0
    IBI_den = NDBI + (NDVI + MNDWI) / 2.0
    IBI = safe_divide(IBI_num, IBI_den)
    result.update(compute_stats(IBI, 'IBI'))

    # 仮説3: 裸地・建設中エリア
    BSI_num = (B11 + B4) - (B8 + B2)
    BSI_den = (B11 + B4) + (B8 + B2)
    BSI = safe_divide(BSI_num, BSI_den)
    result.update(compute_stats(BSI, 'BSI'))

    NBSI = safe_divide(B11 - B8, B11 + B8)  # NDEI とも同義
    result.update(compute_stats(NBSI, 'NBSI'))

    # 仮説4: 水域
    NDWI = safe_divide(B3 - B8, B3 + B8)
    result.update(compute_stats(NDWI, 'NDWI'))

    result.update(compute_stats(MNDWI, 'MNDWI'))

    # 仮説6: 地質・土壌
    Clay = safe_divide(B11, B12)
    result.update(compute_stats(Clay, 'Clay'))

    IronOxide = safe_divide(B4, B2)
    result.update(compute_stats(IronOxide, 'IronOxide'))

    Carbonate = safe_divide(B11, B8A)
    result.update(compute_stats(Carbonate, 'Carbonate'))

    return result

In [5]:
# Sentinel-2 の処理（ユニークファイルごとにキャッシュ）
sentinel2_cache = {}
unique_sentinel2 = df['sentinel2_tiff_file_name'].unique()

print(f'Processing {len(unique_sentinel2)} unique Sentinel-2 files...')
for fname in tqdm(unique_sentinel2):
    tif_path = COMPOSITE_DIR / fname
    if not tif_path.exists():
        print(f'WARNING: {tif_path} not found, skipping.')
        sentinel2_cache[fname] = {}
        continue
    sentinel2_cache[fname] = process_sentinel2(tif_path)

print('Done.')

Processing 1024 unique Sentinel-2 files...


  0%|          | 0/1024 [00:00<?, ?it/s]

Done.


In [6]:
def process_viirs(tif_path):
    """
    VIIRS TIF を読み込み、avg_rad バンドの統計量を返す。
    """
    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(np.float64).ravel()  # avg_rad のみ

    return compute_stats(data, 'avg_rad')

In [7]:
# VIIRS の処理（ユニークファイルごとにキャッシュ）
viirs_cache = {}
unique_viirs = df['viirs_tiff_file_name'].unique()

print(f'Processing {len(unique_viirs)} unique VIIRS files...')
for fname in tqdm(unique_viirs):
    tif_path = COMPOSITE_DIR / fname
    if not tif_path.exists():
        print(f'WARNING: {tif_path} not found, skipping.')
        viirs_cache[fname] = {}
        continue
    viirs_cache[fname] = process_viirs(tif_path)

print('Done.')

Processing 1024 unique VIIRS files...


  0%|          | 0/1024 [00:00<?, ?it/s]

Done.


In [8]:
# 特徴量を DataFrame 化して結合
s2_rows = [sentinel2_cache.get(fname, {}) for fname in df['sentinel2_tiff_file_name']]
viirs_rows = [viirs_cache.get(fname, {}) for fname in df['viirs_tiff_file_name']]

s2_df = pd.DataFrame(s2_rows, index=df.index)
viirs_df = pd.DataFrame(viirs_rows, index=df.index)

print(f'Sentinel-2 feature columns: {len(s2_df.columns)}')
print(f'VIIRS feature columns: {len(viirs_df.columns)}')

Sentinel-2 feature columns: 312
VIIRS feature columns: 13


In [9]:
# CSV に結合して保存
df_out = pd.concat([df, s2_df, viirs_df], axis=1)

print(f'Output shape: {df_out.shape}')
print(f'Added columns: {len(df_out.columns) - len(df.columns)}')
print('\nSample new columns:', [c for c in df_out.columns if c not in df.columns][:10])

df_out.to_csv(OUTPUT_CSV_PATH, index=False)
print(f'\nSaved to {OUTPUT_CSV_PATH}')

Output shape: (1024, 347)
Added columns: 325

Sample new columns: ['B1_pct0', 'B1_pct10', 'B1_pct20', 'B1_pct30', 'B1_pct40', 'B1_pct50', 'B1_pct60', 'B1_pct70', 'B1_pct80', 'B1_pct90']

Saved to ../data/evaluation_dataset_88c8c1d9376e4d9b8034f77ca19939b0/evaluation_tabular_no_target-add-features.csv


In [10]:
# 確認
df_check = pd.read_csv(OUTPUT_CSV_PATH)
print(f'Verified shape: {df_check.shape}')
print('NaN counts in new columns (top 10):')
new_cols = [c for c in df_check.columns if c not in df.columns or c in s2_df.columns or c in viirs_df.columns]
nan_counts = df_check[new_cols].isna().sum()
print(nan_counts[nan_counts > 0].head(10))

Verified shape: (1024, 347)
NaN counts in new columns (top 10):
Series([], dtype: int64)
